# NẠP DỮ LIỆU GỐC

In [1]:
import os
import sys
import zipfile
import urllib.request
import pandas as pd
import numpy as np

In [2]:
ROOT = os.path.abspath('..') if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
RAW_DIR = os.path.join(ROOT, 'data', 'raw')
PROC_DIR = os.path.join(ROOT, 'data', 'processed')
os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(PROC_DIR, exist_ok=True)

In [3]:
#Tải bộ dữ liệu UIT-ViSFD nếu chưa có trong thư mục raw
ZIP_URL = 'https://raw.githubusercontent.com/LuongPhan/UIT-ViSFD/main/UIT-ViSFD.zip'
zip_path = os.path.join(RAW_DIR, 'UIT-ViSFD.zip')
if not os.path.exists(os.path.join(RAW_DIR, 'Train.csv')):
    urllib.request.urlretrieve(ZIP_URL, zip_path)
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(RAW_DIR)

In [4]:
os.listdir(RAW_DIR)

['Dev.csv', 'Test.csv', 'Train.csv', 'UIT-ViSFD.zip']

In [5]:
#Đọc tập train, dev và test thô
df_train_raw = pd.read_csv(os.path.join(RAW_DIR, 'Train.csv'))
df_dev_raw = pd.read_csv(os.path.join(RAW_DIR, 'Dev.csv'))
df_test_raw = pd.read_csv(os.path.join(RAW_DIR, 'Test.csv'))

In [6]:
df_train_raw.shape, df_dev_raw.shape, df_test_raw.shape

((7786, 5), (1112, 5), (2224, 5))

In [7]:
df_train_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 7786 entries, 0 to 7785
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   index      7786 non-null   int64
 1   comment    7786 non-null   str  
 2   n_star     7786 non-null   int64
 3   date_time  7786 non-null   str  
 4   label      7786 non-null   str  
dtypes: int64(2), str(3)
memory usage: 2.3 MB


In [8]:
df_train_raw.head()

,index,comment,n_star,date_time,label
0,0,Mới mua máy này Tại thegioididong thốt nốt cảm...,5,2 tuần trước,{CAMERA#Positive};{FEATURES#Positive};{BATTERY...
1,1,Pin kém còn lại miễn chê mua 8/3/2019 tình trạ...,5,14/09/2019,{BATTERY#Negative};{GENERAL#Positive};{OTHERS};
2,2,Sao lúc gọi điện thoại màn hình bị chấm nhỏ nh...,3,17/08/2020,{FEATURES#Negative};
3,3,"Mọi người cập nhật phần mềm lại , nó sẽ bớt tố...",3,29/02/2020,{FEATURES#Negative};{BATTERY#Neutral};{GENERAL...
4,4,"Mới mua Sài được 1 tháng thấy pin rất trâu, Sà...",5,4/6/2020,{BATTERY#Positive};{PERFORMANCE#Positive};{SER...


## QUY ĐỔI NHÃN THUẦN TÚY VÀ ĐA KHÍA CẠNH

In [9]:
#Nạp hàm giải mã nhãn thuần từ package src để tuyệt đối không đọc n_star
from src.labels import ABSA_COLS, ABSA_STATE_VI, ASPECT_GROUPS, parse_visfd_labels


In [10]:
#Thử nghiệm bóc tách nhãn trên 1 dòng mẫu vừa khen vừa chê ở mức 5 sao
tempt = df_train_raw.iloc[1]
tempt['n_star'], tempt['label'], tempt['comment']

(np.int64(5),
 '{BATTERY#Negative};{GENERAL#Positive};{OTHERS};',
 'Pin kém còn lại miễn chê mua 8/3/2019 tình trạng pin còn 88% có ai giống tôi không')

In [11]:
parse_visfd_labels(tempt['label'], tempt['comment'])

{'sentiment': 1,
 'aspect_group': 2,
 'aspects': [0, 0, 1, 0, 1],
 'all_aspects': [2, 4],
 'absa': [0, 0, 1, 0, 3],
 'has_conflict': False,
 'conflict_with_neg': False,
 'conflict_groups': []}

## ÁP DỤNG CHO TOÀN BỘ 3 TẬP DỮ LIỆU

In [12]:
#Áp dụng parse_visfd_labels tạo nhãn v3 và nhãn True ABSA 5x4 (absa_0..4, has_conflict)
for df in [df_train_raw, df_dev_raw, df_test_raw]:
    parsed = [parse_visfd_labels(r['label'], r['comment']) for _, r in df.iterrows()]
    df['sentiment'] = [p['sentiment'] for p in parsed]
    df['aspect_group'] = [p['aspect_group'] for p in parsed]
    for g in range(5):
        df[f'asp_{g}'] = [p['aspects'][g] for p in parsed]
    df['all_aspects'] = [','.join(str(x) for x in p['all_aspects']) for p in parsed]
    for g in range(5):
        df[f'absa_{g}'] = [p['absa'][g] for p in parsed]
    df['has_conflict'] = [bool(p['has_conflict']) for p in parsed]
    df['conflict_with_neg'] = [bool(p['conflict_with_neg']) for p in parsed]
    df['conflict_groups'] = [','.join(str(x) for x in p['conflict_groups']) for p in parsed]


In [13]:
df_train_raw['sentiment'].value_counts().sort_index()

sentiment
0    2520
1    1524
2    3742
Name: count, dtype: int64

In [14]:
#Kiểm tra bảng chéo n_star và sentiment thuần để phát hiện ca mâu thuẫn sao thật
pd.crosstab(df_train_raw['n_star'], df_train_raw['sentiment'], margins=True)

sentiment,0,1,2,All
n_star,,,,
1,1112,134,25,1271
2,427,117,23,567
3,604,308,100,1012
4,258,444,564,1266
5,119,521,3030,3670
All,2520,1524,3742,7786


In [15]:
df_train_raw['aspect_group'].value_counts().sort_index()

aspect_group
0    3195
1     939
2    1642
3    1097
4     913
Name: count, dtype: int64

In [16]:
#Thống kê số lượng nhãn đa khía cạnh trên 5 nhóm
df_train_raw[['asp_0', 'asp_1', 'asp_2', 'asp_3', 'asp_4']].sum()

asp_0    5503
asp_1    2699
asp_2    3604
asp_3    3515
asp_4    6160
dtype: int64

In [17]:
#Thống kê phân bố 4 trạng thái ABSA (None/Neg/Neu/Pos) và số ca xung đột nội bộ nhóm
absa_dist = pd.DataFrame({ASPECT_GROUPS[g]: df_train_raw[f'absa_{g}'].value_counts() for g in range(5)}).fillna(0).astype(int).sort_index()
absa_dist.index = [f'{i}_{ABSA_STATE_VI[i]}' for i in absa_dist.index]
for name, d in [('train', df_train_raw), ('dev', df_dev_raw), ('test', df_test_raw)]:
    print(f"{name}: has_conflict={int(d['has_conflict'].sum())}, conflict_with_neg={int(d['conflict_with_neg'].sum())}")
absa_dist


train: has_conflict=1025, conflict_with_neg=609
dev: has_conflict=128, conflict_with_neg=83
test: has_conflict=286, conflict_with_neg=141


,PERF_FEAT,SCREEN_CAM,BATTERY,PRICE_SER,DESIGN_GEN
0_Không có,2283,5087,4182,4271,2327
1_Tiêu cực,2706,945,1228,762,1208
2_Trung lập,486,307,349,1190,346
3_Tích cực,2311,1447,2027,1563,3905


## LƯU VÀ GỘP DỮ LIỆU

In [18]:
#Đổi tên cột comment thành sentence để đồng bộ toàn pipeline
for df in [df_train_raw, df_dev_raw, df_test_raw]:
    df.rename(columns={'comment': 'sentence'}, inplace=True)

In [19]:
df_train_raw.head()

,index,sentence,n_star,date_time,label,sentiment,aspect_group,asp_0,asp_1,asp_2,...,asp_4,all_aspects,absa_0,absa_1,absa_2,absa_3,absa_4,has_conflict,conflict_with_neg,conflict_groups
0,0,Mới mua máy này Tại thegioididong thốt nốt cảm...,5,2 tuần trước,{CAMERA#Positive};{FEATURES#Positive};{BATTERY...,2,2,1,1,1,...,1,"0,1,2,3,4",3,3,3,3,3,False,False,
1,1,Pin kém còn lại miễn chê mua 8/3/2019 tình trạ...,5,14/09/2019,{BATTERY#Negative};{GENERAL#Positive};{OTHERS};,1,2,0,0,1,...,1,"2,4",0,0,1,0,3,False,False,
2,2,Sao lúc gọi điện thoại màn hình bị chấm nhỏ nh...,3,17/08/2020,{FEATURES#Negative};,0,0,1,0,0,...,0,0,1,0,0,0,0,False,False,
3,3,"Mọi người cập nhật phần mềm lại , nó sẽ bớt tố...",3,29/02/2020,{FEATURES#Negative};{BATTERY#Neutral};{GENERAL...,1,0,1,0,1,...,1,"0,2,4",1,0,2,0,2,False,False,
4,4,"Mới mua Sài được 1 tháng thấy pin rất trâu, Sà...",5,4/6/2020,{BATTERY#Positive};{PERFORMANCE#Positive};{SER...,2,3,1,0,1,...,0,"0,2,3",3,0,3,1,0,False,False,


In [20]:
#Lưu 3 tập dữ liệu ra thư mục processed
df_train_raw.to_csv(os.path.join(PROC_DIR, 'train_raw.csv'), index=False, encoding='utf-8-sig')
df_dev_raw.to_csv(os.path.join(PROC_DIR, 'dev_raw.csv'), index=False, encoding='utf-8-sig')
df_test_raw.to_csv(os.path.join(PROC_DIR, 'test_raw.csv'), index=False, encoding='utf-8-sig')

In [21]:
#Gộp cả 3 tập kèm cột split thành visfd_merged_raw.csv
df_merged = pd.concat([
    df_train_raw.assign(split='train'),
    df_dev_raw.assign(split='dev'),
    df_test_raw.assign(split='test'),
], ignore_index=True)
df_merged.to_csv(os.path.join(PROC_DIR, 'visfd_merged_raw.csv'), index=False, encoding='utf-8-sig')
df_merged.shape

(11122, 22)

In [22]:
df_merged['split'].value_counts()

split
train    7786
test     2224
dev      1112
Name: count, dtype: int64

In [23]:
os.listdir(PROC_DIR)

['actionable_insights.csv',
 'dev_clean.csv',
 'dev_pred_phobert.csv',
 'dev_raw.csv',
 'sample_excel_predicted.xlsx',
 'test_clean.csv',
 'test_pred_baseline.csv',
 'test_pred_phobert.csv',
 'test_raw.csv',
 'train_clean.csv',
 'train_raw.csv',
 'visfd_merged_raw.csv']